In [1]:
# =========================================================
# Bronze Layer Validation Notebook
# File: notebooks/bronze_validation.ipynb
# =========================================================


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

import yaml
import os

In [3]:
spark = SparkSession.builder.appName("Bronze Layer Validation").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 20:13:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
# =========================================================
# CELL 3 — LOAD CONFIG FILES
# =========================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

with open("../configs/paths.yaml", "r") as file:
    paths_config = yaml.safe_load(file)

with open("../configs/datasets.yaml", "r") as file:
    datasets_config = yaml.safe_load(file)

BRONZE_PATH = str(PROJECT_ROOT / paths_config["bronze_path"])

In [11]:
# =========================================================
# CELL 4 — DISPLAY AVAILABLE BRONZE DATASETS
# =========================================================
print("=" * 60)
print("AVAILABLE BRONZE DATASETS")
print("=" * 60)

for dataset in datasets_config["datasets"]:
    print(f"- {dataset['name']}")

print(BRONZE_PATH,"orders")




AVAILABLE BRONZE DATASETS
- orders
- order_items
- customers
- sellers
- products
- reviews
- payments
- geolocation
- mql
- closed_deals
/Users/hamid/Desktop/Olist_Seller_Intelligence_Platform/data/bronze orders


In [12]:
# =========================================================
# CELL 5 — LOAD SAMPLE BRONZE DATASET
# =========================================================

dataset_name = "orders"

orders_df = spark.read.parquet(
    os.path.join(BRONZE_PATH, dataset_name)
)

print(f"Loaded dataset: {dataset_name}")



Loaded dataset: orders


In [13]:
# =========================================================
# CELL 6 — DISPLAY SCHEMA
# =========================================================

print("=" * 60)
print("SCHEMA VALIDATION")
print("=" * 60)

orders_df.printSchema()

SCHEMA VALIDATION
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- source_file: string (nullable = true)



In [14]:
# =========================================================
# CELL 8 — VALIDATE METADATA COLUMNS
# =========================================================

print("=" * 60)
print("METADATA COLUMN VALIDATION")
print("=" * 60)

required_metadata_columns = [
    "ingestion_timestamp",
    "ingestion_date",
    "source_file"
]

existing_columns = orders_df.columns

for column_name in required_metadata_columns:

    if column_name in existing_columns:
        print(f"[PASS] {column_name} exists")
    else:
        print(f"[FAIL] {column_name} missing")


METADATA COLUMN VALIDATION
[PASS] ingestion_timestamp exists
[PASS] ingestion_date exists
[PASS] source_file exists


In [15]:
# =========================================================
# CELL 9 — ROW COUNT VALIDATION
# =========================================================

print("=" * 60)
print("ROW COUNT VALIDATION")
print("=" * 60)

row_count = orders_df.count()

print(f"Total Rows: {row_count}")

ROW COUNT VALIDATION
Total Rows: 99441


In [16]:
# =========================================================
# CELL 10 — NULL VALUE ANALYSIS
# =========================================================

print("=" * 60)
print("NULL VALUE ANALYSIS")
print("=" * 60)

null_counts = orders_df.select([
    (
        col(column_name).isNull().cast("int")
    ).alias(column_name)

    for column_name in orders_df.columns
])

null_counts.show()

NULL VALUE ANALYSIS
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+--------------+-----------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|ingestion_timestamp|ingestion_date|source_file|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+--------------+-----------+
|       0|          0|           0|                       0|                0|                           0|                            0|                            0|                  0|             0|          0|
|       0|          0|           0|                       0|                0|                           0|             

In [17]:
# =========================================================
# CELL 11 — TIMESTAMP VALIDATION
# =========================================================

print("=" * 60)
print("TIMESTAMP VALIDATION")
print("=" * 60)

timestamp_columns = [
    field.name
    for field in orders_df.schema.fields
    if field.dataType.typeName() == "timestamp"
]

print("Timestamp Columns:")

for column_name in timestamp_columns:
    print(f"- {column_name}")

TIMESTAMP VALIDATION
Timestamp Columns:
- order_purchase_timestamp
- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date
- order_estimated_delivery_date
- ingestion_timestamp


In [18]:
# =========================================================
# CELL 12 — ZIP PREFIX VALIDATION
# =========================================================
# IMPORTANT:
# Ensure ZIP prefixes were not corrupted
# Example:
# 01037 -> 1037 problem

print("=" * 60)
print("ZIP PREFIX VALIDATION")
print("=" * 60)

customers_df = spark.read.parquet(
    os.path.join(BRONZE_PATH, "customers")
)

customers_df.select(
    "customer_zip_code_prefix"
).show(10, truncate=False)

ZIP PREFIX VALIDATION
+------------------------+
|customer_zip_code_prefix|
+------------------------+
|68590                   |
|15056                   |
|13302                   |
|45638                   |
|29700                   |
|18055                   |
|44054                   |
|9812                    |
|70673                   |
|22793                   |
+------------------------+
only showing top 10 rows


In [19]:
# =========================================================
# CELL 13 — PARQUET FILE VALIDATION
# =========================================================

print("=" * 60)
print("PARQUET FILE VALIDATION")
print("=" * 60)

try:

    test_df = spark.read.parquet(
        os.path.join(BRONZE_PATH, "order_items")
    )

    print("[PASS] Parquet files readable")

except Exception as error:

    print("[FAIL] Parquet read error")
    print(error)

PARQUET FILE VALIDATION
[PASS] Parquet files readable


In [20]:
# =========================================================
# CELL 14 — MULTI-DATASET VALIDATION LOOP
# =========================================================

print("=" * 60)
print("BRONZE DATASET VALIDATION SUMMARY")
print("=" * 60)

for dataset in datasets_config["datasets"]:

    dataset_name = dataset["name"]

    try:

        df = spark.read.parquet(
            os.path.join(BRONZE_PATH, dataset_name)
        )

        print(f"[PASS] {dataset_name}")
        print(f"Rows: {df.count()}")
        print(f"Columns: {len(df.columns)}")
        print("-" * 40)

    except Exception as error:

        print(f"[FAIL] {dataset_name}")
        print(error)

BRONZE DATASET VALIDATION SUMMARY
[PASS] orders
Rows: 99441
Columns: 11
----------------------------------------
[PASS] order_items
Rows: 112650
Columns: 10
----------------------------------------
[PASS] customers
Rows: 99441
Columns: 8
----------------------------------------
[PASS] sellers
Rows: 3095
Columns: 7
----------------------------------------
[PASS] products
Rows: 32951
Columns: 12
----------------------------------------
[PASS] reviews
Rows: 104162
Columns: 10
----------------------------------------
[PASS] payments
Rows: 103886
Columns: 8
----------------------------------------
[PASS] geolocation
Rows: 1000163
Columns: 8
----------------------------------------
[PASS] mql
Rows: 8000
Columns: 7
----------------------------------------
[PASS] closed_deals
Rows: 842
Columns: 17
----------------------------------------


In [21]:
# =========================================================
# CELL 15 — INGESTION METADATA INSPECTION
# =========================================================

print("=" * 60)
print("INGESTION METADATA SAMPLE")
print("=" * 60)

orders_df.select(
    "ingestion_timestamp",
    "ingestion_date",
    "source_file"
).show(5, truncate=False)

INGESTION METADATA SAMPLE
+--------------------------+--------------+------------------------+
|ingestion_timestamp       |ingestion_date|source_file             |
+--------------------------+--------------+------------------------+
|2026-05-20 18:36:53.633777|2026-05-20    |olist_orders_dataset.csv|
|2026-05-20 18:36:53.633777|2026-05-20    |olist_orders_dataset.csv|
|2026-05-20 18:36:53.633777|2026-05-20    |olist_orders_dataset.csv|
|2026-05-20 18:36:53.633777|2026-05-20    |olist_orders_dataset.csv|
|2026-05-20 18:36:53.633777|2026-05-20    |olist_orders_dataset.csv|
+--------------------------+--------------+------------------------+
only showing top 5 rows


In [22]:
# =========================================================
# CELL 16 — FINAL VALIDATION SUMMARY
# =========================================================

print("=" * 60)
print("BRONZE LAYER VALIDATION COMPLETED")
print("=" * 60)

print("""
Validation Checklist:

[✓] Bronze parquet readable
[✓] Schemas validated
[✓] Metadata columns validated
[✓] Timestamp columns inspected
[✓] ZIP prefixes reviewed
[✓] Row counts validated
[✓] Multi-dataset validation completed

Bronze Layer Status:
READY FOR SILVER TRANSFORMATION LAYER
""")


BRONZE LAYER VALIDATION COMPLETED

Validation Checklist:

[✓] Bronze parquet readable
[✓] Schemas validated
[✓] Metadata columns validated
[✓] Timestamp columns inspected
[✓] ZIP prefixes reviewed
[✓] Row counts validated
[✓] Multi-dataset validation completed

Bronze Layer Status:
READY FOR SILVER TRANSFORMATION LAYER



In [23]:
# =========================================================
# CELL 17 — STOP SPARK SESSION
# =========================================================

spark.stop()